In [4]:
!pip install pytorch-forecasting pytorch-lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.8/399.8 kB 10.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 13.7 MB/s eta 0:00:00


In [2]:
!pip install py7zr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 14.2 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import torch
import gc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import py7zr
from sklearn.metrics import mean_squared_log_error
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer,TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_forecasting.data.encoders import NaNLabelEncoder

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
def unpack_data(file_path,extract_path):
    with py7zr.SevenZipFile(file_path, mode='r') as archive:
        archive.extractall(path=extract_path)
        print("Распаковка завершена")

arch_path = '/kaggle/input/competitions/favorita-grocery-sales-forecasting/'
extract_path = '/kaggle/working/'

In [8]:
unpack_data(str(arch_path + "holidays_events.csv.7z"),extract_path)
unpack_data(str(arch_path + "items.csv.7z"),extract_path)
unpack_data(str(arch_path + "oil.csv.7z"),extract_path)
unpack_data(str(arch_path + "stores.csv.7z"),extract_path)

Распаковка завершена
Распаковка завершена
Распаковка завершена
Распаковка завершена


In [9]:
holidays=pd.read_csv('/kaggle/working/holidays_events.csv')
items = pd.read_csv("/kaggle/working/items.csv")
oil = pd.read_csv("/kaggle/working/oil.csv")
stores = pd.read_csv("/kaggle/working/stores.csv")

Таблицы мы посмотрели в предыдущей серии про бейзлайн и catboost. Поэтому просто обработаем их.

In [39]:
holidays.date = pd.to_datetime(holidays.date)

items.item_nbr = items.item_nbr.astype('int32')
items['class'] = items['class'].astype('int8')
items.perishable = items.perishable.astype('int8')

oil.date = pd.to_datetime(oil.date)
oil.dcoilwtico = oil.dcoilwtico.astype('float32')

stores.store_nbr = stores.store_nbr.astype('int8')
stores.cluster = stores.cluster.astype('int8')
stores = stores.rename(columns={'type':'type_s'})

oil = oil.set_index('date').sort_index()
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear')
oil = oil.reset_index()
oil.dropna(inplace=True)

In [11]:
def rmsle(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))

In [12]:
path_to_train = '/kaggle/input/datasets/nataliyakleymenova/ts-hw3/train.csv'

Так как датасет огромный, примерно 125 млн. строк, все считать мы не можем. Поэтому будем рассматривать только данные с января 2017 года. Одно порцией тоже считать не получается, поэтому будем считывать кусочками и сразу приводить у минимально возможным типам.

In [90]:
start_date = '2017-01-01'
chunk_size = 1000000

chunks = []
for chunk in pd.read_csv(path_to_train, 
                         index_col=0,
                         parse_dates=['date'],
                         chunksize=chunk_size):
    chunk.store_nbr = chunk.store_nbr.astype('int8')
    chunk.item_nbr = chunk.item_nbr.astype('int32')
    chunk.unit_sales = chunk.unit_sales.astype('float32')
    chunk['onpromotion'] =chunk['onpromotion'].fillna(0)
    chunk.onpromotion = chunk.onpromotion.astype('int8')
    chunk = chunk[chunk['date'] >= start_date]
    if not chunk.empty:
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
df['unit_sales'] = df['unit_sales'].clip(lower=0)

/tmp/ipykernel_55/3339555760.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path_to_train,


Теперь возьмем из этих данных несколько десятков тысяч самых длинных рядов.

In [91]:
row_num = 20000
series_length = df.groupby(['store_nbr', 'item_nbr']).size().reset_index(name='count')
series_length = series_length.sort_values('count', ascending=False)
top_series = series_length.head(row_num)[['store_nbr', 'item_nbr']]

df = df.merge(top_series, on=['store_nbr', 'item_nbr'], how='inner')

Так ряды нерегулярные, дозаполним их.

In [92]:
min_date = df['date'].min()
max_date = df['date'].max()
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')

filled_groups = []

for (store, item), group in tqdm(df.groupby(['store_nbr', 'item_nbr']), desc="Обработка групп"):
    group = group.set_index('date')
    group = group.reindex(all_dates)
    group['store_nbr'] = store
    group['item_nbr'] = item
    group['unit_sales'] = group['unit_sales'].fillna(0)
    group['onpromotion'] = group['onpromotion'].fillna(0)
    group = group.reset_index().rename(columns={'index': 'date'})
    filled_groups.append(group)

df = pd.concat(filled_groups, ignore_index=True)

Обработка групп: 100%|██████████| 20000/20000 [00:32<00:00, 612.28it/s]


In [93]:
del all_dates,filled_groups,chunks
gc.collect()

0

In [94]:
df['year'] = df['date'].dt.year.astype('int16')
df['month'] = df['date'].dt.month.astype('int8')
df['day'] = df['date'].dt.day.astype('int8')
df['dayofweek'] = df['date'].dt.dayofweek.astype('int8')
df['weekend'] = (df['dayofweek'] >= 5).astype('boolean')

for lag in [1, 7, 14, 28]:
        df[f'lag_{lag}'] = df.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(lag)

for col in ['lag_1', 'lag_7', 'lag_14', 'lag_28']:
    df[col] = df[col].fillna(0)

In [95]:
df = df.merge(items[['item_nbr', 'family', 'perishable', 'class']], on='item_nbr', how='left')
df = df.merge(stores, on='store_nbr', how='left')

national_holidays = holidays[holidays['locale'] == 'National']['date'].unique()
df['is_national_holiday'] = df['date'].isin(national_holidays).astype('boolean')

mask = (oil['date'] >= '2016-01-01') & (oil['date'] <= '2016-12-31')
oil_period = oil.loc[mask]
oil_mean_price = oil_period['dcoilwtico'].mean()
df = df.merge(oil, on='date', how='left')
df['dcoilwtico'] = df['dcoilwtico'].fillna(oil_mean_price)


static_features = ['family', 'perishable', 'city', 'state', 'type_s', 'cluster']

for col in static_features:
        df[col] = df[col].fillna('unknown')
df['class'] = df['class'].fillna(0)


In [96]:
df["time_idx"] = (df["date"] - df["date"].min()).dt.days
df['series_id'] = df['store_nbr'].astype(str) + '_' + df['item_nbr'].astype(str)
df['log_sales'] = np.log1p(df['unit_sales'])

In [97]:
max_encoder_length = 30
max_prediction_length = 16

In [98]:
val_end = df['date'].max()
val_start = val_end - pd.DateOffset(days=max_prediction_length - 1)  # первый день валидации
val_start_history = val_start - pd.DateOffset(days=max_encoder_length)
train_tft = df[df['date'] < val_start]
val_tft = df[(df['date'] >= val_start_history) ]

In [99]:
del df
gc.collect()

44

In [100]:
static_categoricals = ['series_id', 'store_nbr', 'item_nbr', 'family',
                      'city', 'state', 'type_s', 'cluster', 'perishable']
static_reals = [] 

known_future_categoricals = ['dayofweek', 'month', 'is_national_holiday']
known_future_reals = ['day']#, 'days_to_holiday']  # день месяца и дни до праздника
past_categoricals = []  
past_reals = ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'dcoilwtico']
all_categoricals = static_categoricals + known_future_categoricals

for col in all_categoricals:
        train_tft[col] = train_tft[col].astype(str).astype("category")
        val_tft[col] = val_tft[col].astype(str).astype("category")

In [101]:
training = TimeSeriesDataSet(
    train_tft,
    time_idx="time_idx",
    target="log_sales",

    group_ids=["series_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=static_categoricals,
    time_varying_known_categoricals=known_future_categoricals,
    time_varying_known_reals=known_future_reals,
    time_varying_unknown_reals=past_reals,
    categorical_encoders={
        'store_nbr': NaNLabelEncoder(add_nan=True),
        'item_nbr': NaNLabelEncoder(add_nan=True),
        'series_id': NaNLabelEncoder(add_nan=True),
        'series_id': NaNLabelEncoder(add_nan=True),
        'family': NaNLabelEncoder(add_nan=True),
        'city': NaNLabelEncoder(add_nan=True),
        'state': NaNLabelEncoder(add_nan=True),
        'cluster': NaNLabelEncoder(add_nan=True),
        'month': NaNLabelEncoder(add_nan=True)},
    target_normalizer=GroupNormalizer(
        groups=["series_id"]    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True
)

In [102]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    val_tft,
    predict=True,
    stop_randomization=True
)

/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/data/encoders.py:402: UserWarning: Found 1 unknown classes which were set to NaN
  warnings.warn(


In [103]:
batch_size = 1024

train_loader = training.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=4,
    pin_memory=True)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=4,
    pin_memory=True)

In [104]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=32,
    output_size=7,
    loss=QuantileLoss(),
    log_interval=10
).to(device)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [105]:
def move_batch_to_device(batch, device):

    if torch.is_tensor(batch):
        return batch.to(device)
    elif isinstance(batch, dict):
        return {k: move_batch_to_device(v, device) for k, v in batch.items()}
    elif isinstance(batch, (tuple, list)):
        return type(batch)(move_batch_to_device(v, device) for v in batch)
    else:
        return batch

In [106]:
optimizer = torch.optim.Adam(tft.parameters(), lr=1e-3)

In [107]:
num_epochs = 10

for epoch in range(num_epochs):
    tft.train()
    train_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Эпоха {epoch+1}/{num_epochs} - Обучение", leave=False):
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad()
        d,target_weight = batch
        y_pred = tft(d)  
        target,weight = target_weight
        loss = tft.loss(y_pred[0], target)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(target)
  
    train_loss /= len(train_loader.dataset)

    tft.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Эпоха {epoch+1}/{num_epochs} - Валидация", leave=False):
            batch = move_batch_to_device(batch, device)
            d,target_weight = batch
            y_pred = tft(d)
            target,weight = target_weight    
            loss = tft.loss(y_pred[0], target)
            val_loss += loss.item() * len(target)

    val_loss /= len(val_loader.dataset)
    print(f"Эпоха {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Эпоха 1/10 | Train Loss: 0.2434 | Val Loss: 0.2178


Эпоха 2/10 | Train Loss: 0.2080 | Val Loss: 0.2138


Эпоха 3/10 | Train Loss: 0.2033 | Val Loss: 0.2120


Эпоха 4/10 | Train Loss: 0.1996 | Val Loss: 0.2129


Эпоха 5/10 | Train Loss: 0.1965 | Val Loss: 0.2132


Эпоха 6/10 | Train Loss: 0.1937 | Val Loss: 0.2122


Эпоха 7/10 | Train Loss: 0.1912 | Val Loss: 0.2134


Эпоха 8/10 | Train Loss: 0.1890 | Val Loss: 0.2142


Эпоха 9/10 | Train Loss: 0.1870 | Val Loss: 0.2153


Эпоха 10/10 | Train Loss: 0.1852 | Val Loss: 0.2159


In [108]:
predictions = tft.predict(val_loader, return_y=True, trainer_kwargs=dict(accelerator="gpu"))
pred_log = predictions.output.cpu().numpy() 
true_log = predictions.y[0].cpu().numpy()    

pred_sales = np.expm1(pred_log)
true_sales = np.expm1(true_log)

rmsle_tft = rmsle(true_sales, pred_sales)
print("TFT RMSLE: ",rmsle_tft)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


TFT RMSLE:  0.5422556
